In [ ]:
import os
import sys
import argparse
import pickle
import pdb
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from imblearn.over_sampling import SMOTE, BorderlineSMOTE
sys.path.append("../Utils")
from phosfate_utils import *

In [3]:
# from torchvision import datasets, transforms
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cuda device


In [4]:
def create_parser():
    parser = argparse.ArgumentParser(
        description="Generate t-SNE plot with specified parameters"
        )
    parser.add_argument(
        "--ion_symbols",
        type=str,
        help="In case of multi-ion mode, specify the list of ions you want to plot. Default is just K.",
        )
    parser.add_argument(
        "--input_file",
        type=str,
        default=None,
        required=False,
        help="Load the arguments from this input file",
        )
    parser.add_argument(
        "--ion_names",
        type=str,
        help="Specify ion name in a list. Default is just Potassium",
        )
    parser.add_argument(
        "--distance",
        type=int,
        default=5.0,
        help="Specify distance (e.g., 4)",
        )

    parser.add_argument(
        "--number_of_embeddings",
        type=int,
        default=None,
        required=False,
        help="Specify number of embeddings in case of importing specific dataset sizes only.",
        )
    parser.add_argument(
        "--logfile",
        type=str,
        default="Build23-for-LS.log",
        # required=True,
        help="Specify logfile name. Default is Build23-for-LS.log",
        )
    parser.add_argument(
        "--eta",
        type=float,
        default=3,
        # required=True,
        help="Specify logfile name. Default is Build23-for-LS.log",
        )
    return parser

In [5]:
# Read parser and generate ion_names_in_a_list
parser = create_parser()
args = parser.parse_args("")
embeddings_data = []
ion_names_in_a_list = ["Phosphate", "Sulfate", "Chloride", "Nitrate", "Carbonate"]
ion_symbols_in_a_list = ["PO4", "SO4", "CL", "NO3", "CO3"]

In [ ]:
# Load the data into embeddings_data variable
for ion_name in ion_names_in_a_list:
    with open(f"../Features_extraction/esm2/Distance-{args.distance}/{ion_name}/{ion_name}BindingSiteEmbeddings-Distance{args.distance}angstroms-complete.pkl", "rb") as handle:
        pickle_data = pickle.load(handle)
        embeddings_data.append(pickle_data)

In [ ]:
# Edit the next line of code to add/remove descriptors from the embeddings_data variable
site_num, pdb_name, pdb_id, CN_atom, CN_residues, averaged_tensor_embedding, ion_data = [],[],[],[],[],[],[]
stacked_tensorX0 = torch.tensor([])

for i,_ in enumerate(embeddings_data):
    tensors_for_current_ion = []
    labels_for_current_ion = []
    for j,item in enumerate(embeddings_data[i]):
        site_num.append(item[0])
        pdb_name.append(item[1])
        pdb_id.append(item[2])
        CN_atom.append(item[3])
        CN_residues.append(item[4])
        averaged_tensor_embedding.append(item[5])
        tensors_for_current_ion.append(item[5])
        labels_for_current_ion.append(ion_symbols_in_a_list[i])
        ion_data.append(ion_symbols_in_a_list[i])
        
    stacked_tensorX0 = torch.cat((stacked_tensorX0, torch.stack(tensors_for_current_ion)), dim=0)
        
unique,counts = np.unique(ion_data,return_counts=True)
before = dict(zip(unique,counts))
print(f"before: {before}")

before: {np.str_('CL'): np.int64(12622), np.str_('CO3'): np.int64(205), np.str_('NO3'): np.int64(1654), np.str_('PO4'): np.int64(6089), np.str_('SO4'): np.int64(49914)}


In [8]:
# Filter the unique binding sites into variables: stacked_tensorX0 and y_combined_filtered
_, stacked_tensorX0_unique_indices = np.unique(stacked_tensorX0, axis=0, return_index=True)
stacked_tensorX0_unique= stacked_tensorX0[sorted(stacked_tensorX0_unique_indices)]
site_num_unique = np.array(site_num)[sorted(stacked_tensorX0_unique_indices)]
pdb_name_unique = np.array(pdb_name)[sorted(stacked_tensorX0_unique_indices)]
pdb_id_unique = np.array(pdb_id)[sorted(stacked_tensorX0_unique_indices)]
CN_atom_unique = np.array(CN_atom)[sorted(stacked_tensorX0_unique_indices)]
CN_residues_unique = np.array(CN_residues)[sorted(stacked_tensorX0_unique_indices)]
ion_data_unique = np.array(ion_data)[sorted(stacked_tensorX0_unique_indices)]
averaged_tensor_embedding_unique = np.stack(averaged_tensor_embedding)[sorted(stacked_tensorX0_unique_indices)]
stacked_tensorX1 = stacked_tensorX0_unique.reshape(stacked_tensorX0_unique.shape[0], stacked_tensorX0_unique.shape[1], 1)

y_combined = np.array([[0] if item == 'PO4' else [1] if item == 'SO4' else [2] if item == 'CL' else [3] if item == 'NO3' else [4] if item == 'CO3' else [5] for item in ion_data])
y_combined_filtered = y_combined[sorted(stacked_tensorX0_unique_indices)]

unique,counts = np.unique(ion_data_unique,return_counts=True)
after = dict(zip(unique,counts))
print(f"after: {after}")

diff = {}
for key in before.keys():
    diff[key] = before[key]-after[key]
print(f"diff: {diff}")

after: {np.str_('CL'): np.int64(10535), np.str_('CO3'): np.int64(173), np.str_('NO3'): np.int64(1503), np.str_('PO4'): np.int64(5250), np.str_('SO4'): np.int64(42423)}
diff: {np.str_('CL'): np.int64(2087), np.str_('CO3'): np.int64(32), np.str_('NO3'): np.int64(151), np.str_('PO4'): np.int64(839), np.str_('SO4'): np.int64(7491)}


In [9]:
unique, counts = np.unique(y_combined_filtered, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"Class {cls}: {cnt} samples")

Class 0: 5250 samples
Class 1: 42423 samples
Class 2: 10535 samples
Class 3: 1503 samples
Class 4: 173 samples


In [ ]:
# Convert data to numpy if needed
X_plot = stacked_tensorX0_unique.cpu().numpy() if isinstance(stacked_tensorX0_unique, torch.Tensor) else stacked_tensorX0_unique
y_plot = y_combined_filtered.ravel()

# Compute t-SNE
print("Computing t-SNE...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_plot)

# Define colors for each class
class_colors = {
    0: "#1f77b4",  # Blue - PO4
    1: "#ff7f0e",  # Orange - SO4
    2: "#2ca02c",  # Green - CL
    3: "#d62728",  # Red - NO3
    4: "#9467bd",  # Purple - CO3
}

class_names = {
    0: "PO4",
    1: "SO4",
    2: "CL",
    3: "NO3",
    4: "CO3"
}

# Plot
plt.figure(figsize=(10, 8), dpi=300)

for cls in sorted(np.unique(y_plot)):
    mask = (y_plot == cls)
    plt.scatter(
        X_tsne[mask, 0],
        X_tsne[mask, 1],
        c=class_colors[cls],
        label=class_names[cls],
        s=30,
        alpha=0.7,
        edgecolors='black',
        linewidth=0.3
    )

plt.xlabel("t-SNE 1", fontsize=12)
plt.ylabel("t-SNE 2", fontsize=12)
plt.title("t-SNE Projection of Ion Binding Sites", fontsize=14, fontweight='bold')
plt.legend(title="Ion Type", fontsize=10, title_fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ t-SNE plot completed!")

# SECTION A: Save dataset.pt (NO SMOTE)


In [ ]:
# ------------------------------------------------------------
# Load dataset
# ------------------------------------------------------------
features = stacked_tensorX0_unique
labels = y_combined_filtered

# Convert to numpy for splitting
if isinstance(features, torch.Tensor):
    X = features.detach().cpu().numpy()
else:
    X = np.asarray(features)

y = np.asarray(labels).ravel().astype(int)

print("Original dataset shape:", X.shape, y.shape)
print("Class distribution:", dict(zip(*np.unique(y, return_counts=True))))

# 80-10-10 split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=True,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    shuffle=True,
    stratify=y_temp,
    random_state=42
)

# Convert back to torch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

# Save location
data_save_location = "../Data/multiclass_data_no_SMOTE"
check_dir(data_save_location)

torch.save({'X_train': X_train, 'y_train': y_train}, f'{data_save_location}/training_data.pt')
torch.save({'X_val': X_val, 'y_val': y_val}, f'{data_save_location}/validation_data.pt')
torch.save({'X_test': X_test, 'y_test': y_test}, f'{data_save_location}/test_data.pt')

print("✅ Saved NO-SMOTE dataset to:", data_save_location)
print("Train shape:", X_train.shape, y_train.shape)
print("Val shape:  ", X_val.shape, y_val.shape)
print("Test shape: ", X_test.shape, y_test.shape)

print("Train class distribution:", dict(zip(*np.unique(y_train.numpy(), return_counts=True))))
print("Val class distribution:  ", dict(zip(*np.unique(y_val.numpy(), return_counts=True))))
print("Test class distribution: ", dict(zip(*np.unique(y_test.numpy(), return_counts=True))))

Original dataset shape: (59884, 1280) (59884,)
Class distribution: {np.int64(0): np.int64(5250), np.int64(1): np.int64(42423), np.int64(2): np.int64(10535), np.int64(3): np.int64(1503), np.int64(4): np.int64(173)}
✅ Saved NO-SMOTE dataset to: ../Data_split/multiclass_data_no_SMOTE
Train shape: torch.Size([47907, 1280]) torch.Size([47907])
Val shape:   torch.Size([5988, 1280]) torch.Size([5988])
Test shape:  torch.Size([5989, 1280]) torch.Size([5989])
Train class distribution: {np.int64(0): np.int64(4200), np.int64(1): np.int64(33938), np.int64(2): np.int64(8428), np.int64(3): np.int64(1202), np.int64(4): np.int64(139)}
Val class distribution:   {np.int64(0): np.int64(525), np.int64(1): np.int64(4242), np.int64(2): np.int64(1053), np.int64(3): np.int64(151), np.int64(4): np.int64(17)}
Test class distribution:  {np.int64(0): np.int64(525), np.int64(1): np.int64(4243), np.int64(2): np.int64(1054), np.int64(3): np.int64(150), np.int64(4): np.int64(17)}


# SECTION B: SMOTE BEFORE SPLIT + is_synthetic LABEL

In [ ]:
# ------------------------------------------------------------
# Load dataset
# ------------------------------------------------------------
features = stacked_tensorX0_unique
labels   = y_combined_filtered

# Convert to numpy
if isinstance(features, torch.Tensor):
    X = features.detach().cpu().numpy()
else:
    X = np.asarray(features)

y = np.asarray(labels).ravel().astype(int)

print("Original shape:", X.shape)
print("Original class distribution:", dict(zip(*np.unique(y, return_counts=True))))

# ------------------------------------------------------------
# Apply SMOTE
# ------------------------------------------------------------
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X, y)

# ------------------------------------------------------------
# Create is_synthetic flag
# ------------------------------------------------------------
n_original = X.shape[0]
n_total    = X_smote.shape[0]

is_synthetic = np.zeros(n_total, dtype=int)
is_synthetic[n_original:] = 1  # synthetic samples

print("After SMOTE shape:", X_smote.shape)
print("Synthetic samples:", np.sum(is_synthetic))

# ------------------------------------------------------------
# Split (80-10-10)
# ------------------------------------------------------------
X_train, X_temp, y_train, y_temp, syn_train, syn_temp = train_test_split(
    X_smote, y_smote, is_synthetic,
    test_size=0.2,
    stratify=y_smote,
    random_state=42
)

X_val, X_test, y_val, y_test, syn_val, syn_test = train_test_split(
    X_temp, y_temp, syn_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

# ------------------------------------------------------------
# Convert to torch
# ------------------------------------------------------------
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

syn_train = torch.tensor(syn_train, dtype=torch.long)
syn_val   = torch.tensor(syn_val, dtype=torch.long)
syn_test  = torch.tensor(syn_test, dtype=torch.long)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
data_save_location = "../Data/multiclass_data_SMOTE"
check_dir(data_save_location)

torch.save({
    'X_train': X_train,
    'y_train': y_train,
    'is_synthetic': syn_train
}, f'{data_save_location}/training_data.pt')

torch.save({
    'X_val': X_val,
    'y_val': y_val,
    'is_synthetic': syn_val
}, f'{data_save_location}/validation_data.pt')

torch.save({
    'X_test': X_test,
    'y_test': y_test,
    'is_synthetic': syn_test
}, f'{data_save_location}/test_data.pt')

# ------------------------------------------------------------
# Debug prints
# ------------------------------------------------------------
print("✅ Saved SMOTE dataset with synthetic labels")

print("\nTrain synthetic count:", syn_train.sum().item())
print("Val synthetic count:  ", syn_val.sum().item())
print("Test synthetic count: ", syn_test.sum().item())

Original shape: (59884, 1280)
Original class distribution: {np.int64(0): np.int64(5250), np.int64(1): np.int64(42423), np.int64(2): np.int64(10535), np.int64(3): np.int64(1503), np.int64(4): np.int64(173)}
After SMOTE shape: (212115, 1280)
Synthetic samples: 152231
✅ Saved SMOTE dataset with synthetic labels

Train synthetic count: 121768
Val synthetic count:   15232
Test synthetic count:  15231


# SECTION B: SPLIT FIRST, THEN APPLY SMOTE ONLY ON TRAIN + is_synthetic LABEL

In [ ]:
# ------------------------------------------------------------
# Load dataset
# ------------------------------------------------------------
features = stacked_tensorX0_unique
labels   = y_combined_filtered

# Convert to numpy
if isinstance(features, torch.Tensor):
    X = features.detach().cpu().numpy()
else:
    X = np.asarray(features)

y = np.asarray(labels).ravel().astype(int)

print("Original shape:", X.shape)
print("Original class distribution:", dict(zip(*np.unique(y, return_counts=True))))

# ------------------------------------------------------------
# Split first (80-10-10) on ORIGINAL data
# ------------------------------------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=True,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    shuffle=True,
    stratify=y_temp,
    random_state=42
)

print("\nBefore SMOTE:")
print("Train shape:", X_train.shape, "| class distribution:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Val shape:  ", X_val.shape,   "| class distribution:", dict(zip(*np.unique(y_val, return_counts=True))))
print("Test shape: ", X_test.shape,  "| class distribution:", dict(zip(*np.unique(y_test, return_counts=True))))

# ------------------------------------------------------------
# Apply SMOTE ONLY on training set
# ------------------------------------------------------------
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# ------------------------------------------------------------
# Create is_synthetic flag
# Train: original samples = 0, SMOTE-generated = 1
# Val/Test: all original = 0
# ------------------------------------------------------------
n_train_original = X_train.shape[0]
n_train_total    = X_train_smote.shape[0]

is_synthetic_train = np.zeros(n_train_total, dtype=int)
is_synthetic_train[n_train_original:] = 1

is_synthetic_val  = np.zeros(len(y_val), dtype=int)
is_synthetic_test = np.zeros(len(y_test), dtype=int)

print("\nAfter SMOTE on train only:")
print("Train shape:", X_train_smote.shape)
print("Train class distribution:", dict(zip(*np.unique(y_train_smote, return_counts=True))))
print("Synthetic train samples:", np.sum(is_synthetic_train))
print("Synthetic val samples:", np.sum(is_synthetic_val))
print("Synthetic test samples:", np.sum(is_synthetic_test))

# ------------------------------------------------------------
# Convert to torch tensors
# ------------------------------------------------------------
X_train_smote = torch.tensor(X_train_smote, dtype=torch.float32)
X_val         = torch.tensor(X_val, dtype=torch.float32)
X_test        = torch.tensor(X_test, dtype=torch.float32)

y_train_smote = torch.tensor(y_train_smote, dtype=torch.long)
y_val         = torch.tensor(y_val, dtype=torch.long)
y_test        = torch.tensor(y_test, dtype=torch.long)

is_synthetic_train = torch.tensor(is_synthetic_train, dtype=torch.long)
is_synthetic_val   = torch.tensor(is_synthetic_val, dtype=torch.long)
is_synthetic_test  = torch.tensor(is_synthetic_test, dtype=torch.long)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
data_save_location = "../Data/multiclass_data_SMOTE_on_train_only"
check_dir(data_save_location)

torch.save({
    'X_train': X_train_smote,
    'y_train': y_train_smote,
    'is_synthetic': is_synthetic_train
}, f'{data_save_location}/training_data.pt')

torch.save({
    'X_val': X_val,
    'y_val': y_val,
    'is_synthetic': is_synthetic_val
}, f'{data_save_location}/validation_data.pt')

torch.save({
    'X_test': X_test,
    'y_test': y_test,
    'is_synthetic': is_synthetic_test
}, f'{data_save_location}/test_data.pt')

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------
print("\n✅ Saved split-first, SMOTE-on-train-only dataset with is_synthetic flag")
print("Saved to:", data_save_location)

print("\nTorch shapes:")
print("Train:", X_train_smote.shape, y_train_smote.shape, is_synthetic_train.shape)
print("Val:  ", X_val.shape,         y_val.shape,         is_synthetic_val.shape)
print("Test: ", X_test.shape,        y_test.shape,        is_synthetic_test.shape)

Original shape: (59884, 1280)
Original class distribution: {np.int64(0): np.int64(5250), np.int64(1): np.int64(42423), np.int64(2): np.int64(10535), np.int64(3): np.int64(1503), np.int64(4): np.int64(173)}

Before SMOTE:
Train shape: (47907, 1280) | class distribution: {np.int64(0): np.int64(4200), np.int64(1): np.int64(33938), np.int64(2): np.int64(8428), np.int64(3): np.int64(1202), np.int64(4): np.int64(139)}
Val shape:   (5988, 1280) | class distribution: {np.int64(0): np.int64(525), np.int64(1): np.int64(4242), np.int64(2): np.int64(1053), np.int64(3): np.int64(151), np.int64(4): np.int64(17)}
Test shape:  (5989, 1280) | class distribution: {np.int64(0): np.int64(525), np.int64(1): np.int64(4243), np.int64(2): np.int64(1054), np.int64(3): np.int64(150), np.int64(4): np.int64(17)}

After SMOTE on train only:
Train shape: (169690, 1280)
Train class distribution: {np.int64(0): np.int64(33938), np.int64(1): np.int64(33938), np.int64(2): np.int64(33938), np.int64(3): np.int64(33938), n

# SECTION C: SPLIT FIRST, THEN APPLY BORDERLINE-SMOTE ONLY ON TRAIN + is_synthetic LABEL

In [ ]:
# ------------------------------------------------------------
# Load dataset
# ------------------------------------------------------------
features = stacked_tensorX0_unique
labels   = y_combined_filtered

# Convert to numpy
if isinstance(features, torch.Tensor):
    X = features.detach().cpu().numpy()
else:
    X = np.asarray(features)

y = np.asarray(labels).ravel().astype(int)

print("Original shape:", X.shape)
print("Original class distribution:", dict(zip(*np.unique(y, return_counts=True))))

# ------------------------------------------------------------
# Split first (80-10-10) on ORIGINAL data
# ------------------------------------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=True,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    shuffle=True,
    stratify=y_temp,
    random_state=42
)

print("\nBefore Borderline-SMOTE:")
print("Train shape:", X_train.shape, "| class distribution:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Val shape:  ", X_val.shape,   "| class distribution:", dict(zip(*np.unique(y_val, return_counts=True))))
print("Test shape: ", X_test.shape,  "| class distribution:", dict(zip(*np.unique(y_test, return_counts=True))))

# ------------------------------------------------------------
# Apply Borderline-SMOTE ONLY on training set
# kind='borderline-1' is the usual default choice
# You can change to kind='borderline-2' if needed
# ------------------------------------------------------------
bsmote = BorderlineSMOTE(
    kind='borderline-1',
    random_state=42
)
X_train_bsmote, y_train_bsmote = bsmote.fit_resample(X_train, y_train)

# ------------------------------------------------------------
# Create is_synthetic flag
# Train: original samples = 0, generated = 1
# Val/Test: all original = 0
# ------------------------------------------------------------
n_train_original = X_train.shape[0]
n_train_total    = X_train_bsmote.shape[0]

is_synthetic_train = np.zeros(n_train_total, dtype=int)
is_synthetic_train[n_train_original:] = 1

is_synthetic_val  = np.zeros(len(y_val), dtype=int)
is_synthetic_test = np.zeros(len(y_test), dtype=int)

print("\nAfter Borderline-SMOTE on train only:")
print("Train shape:", X_train_bsmote.shape)
print("Train class distribution:", dict(zip(*np.unique(y_train_bsmote, return_counts=True))))
print("Synthetic train samples:", np.sum(is_synthetic_train))
print("Synthetic val samples:", np.sum(is_synthetic_val))
print("Synthetic test samples:", np.sum(is_synthetic_test))

# ------------------------------------------------------------
# Convert to torch tensors
# ------------------------------------------------------------
X_train_bsmote = torch.tensor(X_train_bsmote, dtype=torch.float32)
X_val          = torch.tensor(X_val, dtype=torch.float32)
X_test         = torch.tensor(X_test, dtype=torch.float32)

y_train_bsmote = torch.tensor(y_train_bsmote, dtype=torch.long)
y_val          = torch.tensor(y_val, dtype=torch.long)
y_test         = torch.tensor(y_test, dtype=torch.long)

is_synthetic_train = torch.tensor(is_synthetic_train, dtype=torch.long)
is_synthetic_val   = torch.tensor(is_synthetic_val, dtype=torch.long)
is_synthetic_test  = torch.tensor(is_synthetic_test, dtype=torch.long)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
data_save_location = "../Data/multiclass_data_BorderlineSMOTE_on_train_only"
check_dir(data_save_location)

torch.save({
    'X_train': X_train_bsmote,
    'y_train': y_train_bsmote,
    'is_synthetic': is_synthetic_train
}, f'{data_save_location}/training_data.pt')

torch.save({
    'X_val': X_val,
    'y_val': y_val,
    'is_synthetic': is_synthetic_val
}, f'{data_save_location}/validation_data.pt')

torch.save({
    'X_test': X_test,
    'y_test': y_test,
    'is_synthetic': is_synthetic_test
}, f'{data_save_location}/test_data.pt')

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------
print("\n✅ Saved split-first, Borderline-SMOTE-on-train-only dataset with is_synthetic flag")
print("Saved to:", data_save_location)

print("\nTorch shapes:")
print("Train:", X_train_bsmote.shape, y_train_bsmote.shape, is_synthetic_train.shape)
print("Val:  ", X_val.shape,          y_val.shape,          is_synthetic_val.shape)
print("Test: ", X_test.shape,         y_test.shape,         is_synthetic_test.shape)

Original shape: (59884, 1280)
Original class distribution: {np.int64(0): np.int64(5250), np.int64(1): np.int64(42423), np.int64(2): np.int64(10535), np.int64(3): np.int64(1503), np.int64(4): np.int64(173)}

Before Borderline-SMOTE:
Train shape: (47907, 1280) | class distribution: {np.int64(0): np.int64(4200), np.int64(1): np.int64(33938), np.int64(2): np.int64(8428), np.int64(3): np.int64(1202), np.int64(4): np.int64(139)}
Val shape:   (5988, 1280) | class distribution: {np.int64(0): np.int64(525), np.int64(1): np.int64(4242), np.int64(2): np.int64(1053), np.int64(3): np.int64(151), np.int64(4): np.int64(17)}
Test shape:  (5989, 1280) | class distribution: {np.int64(0): np.int64(525), np.int64(1): np.int64(4243), np.int64(2): np.int64(1054), np.int64(3): np.int64(150), np.int64(4): np.int64(17)}

After Borderline-SMOTE on train only:
Train shape: (169690, 1280)
Train class distribution: {np.int64(0): np.int64(33938), np.int64(1): np.int64(33938), np.int64(2): np.int64(33938), np.int64(